# 15 — Route Risk Scoring

Composite risk score for 5,855 US domestic routes based on Jan–Aug 2025 test set performance.

**Input:** `dataset/FINAL_DATASET.parquet`
**Output:** `models/route_risk_scores.parquet`

**Risk score components:**
- Delay rate (30%) · Avg delay minutes (20%) · Severe delay >60min (15%)
- Cascade score (10%) · Cascade minutes (10%)
- Weather severity origin/dest (5% each) · Congestion (2.5% each)

**Results:** DFW→ALB highest risk (0.605) · OAK→KOA safest (0.045)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

flights = pd.read_parquet('dataset/FINAL_DATASET.parquet',
    columns=['FL_DATE', 'ORIGIN', 'DEST', 'OP_UNIQUE_CARRIER',
             'ARR_DEL15', 'ARR_DELAY', 'DEP_HOUR',
             'cascade_score', 'cascade_delay_minutes',
             'origin_weather_severity', 'dest_weather_severity',
             'real_time_turn_gap', 'airport_fatigue_index',
             'hourly_flight_count', 'dest_hourly_flight_count'])

flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])
test = flights[flights['FL_DATE'] >= '2025-01-01'].copy()

print(f"Test set: {test.shape[0]:,} flights")
print(f"Unique routes: {test.groupby(['ORIGIN', 'DEST']).ngroups:,}")
print("✓ Ready")

Test set: 4,519,126 flights
Unique routes: 6,661
✓ Ready


## Compute Route Risk Scores
Min-max normalize each component then apply weighted sum. Routes with fewer than 50 flights excluded.

In [ ]:
route_stats = test.groupby(['ORIGIN', 'DEST']).agg(
    flights=('ARR_DEL15', 'count'),
    delay_rate=('ARR_DEL15', 'mean'),
    avg_delay_min=('ARR_DELAY', 'mean'),
    severe_delay_pct=('ARR_DELAY', lambda x: (x > 60).mean()),
    avg_cascade_score=('cascade_score', 'mean'),
    avg_cascade_min=('cascade_delay_minutes', 'mean'),
    avg_origin_weather=('origin_weather_severity', 'mean'),
    avg_dest_weather=('dest_weather_severity', 'mean'),
    avg_origin_congestion=('hourly_flight_count', 'mean'),
    avg_dest_congestion=('dest_hourly_flight_count', 'mean'),
).reset_index()

route_stats = route_stats[route_stats['flights'] >= 50].copy()
print(f"Routes with 50+ flights: {len(route_stats):,}")

components = ['delay_rate', 'avg_delay_min', 'severe_delay_pct', 
              'avg_cascade_score', 'avg_cascade_min',
              'avg_origin_weather', 'avg_dest_weather',
              'avg_origin_congestion', 'avg_dest_congestion']

for col in components:
    min_val = route_stats[col].min()
    max_val = route_stats[col].max()
    if max_val > min_val:
        route_stats[f'{col}_norm'] = (route_stats[col] - min_val) / (max_val - min_val)
    else:
        route_stats[f'{col}_norm'] = 0

route_stats['risk_score'] = (
    0.30 * route_stats['delay_rate_norm'] +
    0.20 * route_stats['avg_delay_min_norm'] +
    0.15 * route_stats['severe_delay_pct_norm'] +
    0.10 * route_stats['avg_cascade_score_norm'] +
    0.10 * route_stats['avg_cascade_min_norm'] +
    0.05 * route_stats['avg_origin_weather_norm'] +
    0.05 * route_stats['avg_dest_weather_norm'] +
    0.025 * route_stats['avg_origin_congestion_norm'] +
    0.025 * route_stats['avg_dest_congestion_norm']
)

route_stats['risk_tier'] = pd.cut(route_stats['risk_score'], 
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=['Very Low', 'Low', 'Moderate', 'High', 'Very High'])

print(f"\n{'=' * 75}")
print(f"TOP 20 HIGHEST RISK ROUTES (min 50 flights)")
print(f"{'=' * 75}")
print(f"  {'Route':<14} {'Flights':>8} {'Delay%':>8} {'Avg Min':>8} {'Severe%':>8} {'Risk':>6} {'Tier':<10}")
print(f"  {'-' * 68}")

for _, row in route_stats.nlargest(20, 'risk_score').iterrows():
    route = f"{row['ORIGIN']}→{row['DEST']}"
    print(f"  {route:<14} {row['flights']:>6,}   {row['delay_rate']:>6.1%}   {row['avg_delay_min']:>6.1f}m"
          f"   {row['severe_delay_pct']:>6.1%}   {row['risk_score']:>5.3f} {row['risk_tier']}")

print(f"\n{'=' * 75}")
print(f"TOP 20 SAFEST ROUTES (min 50 flights)")
print(f"{'=' * 75}")
print(f"  {'Route':<14} {'Flights':>8} {'Delay%':>8} {'Avg Min':>8} {'Severe%':>8} {'Risk':>6} {'Tier':<10}")
print(f"  {'-' * 68}")

for _, row in route_stats.nsmallest(20, 'risk_score').iterrows():
    route = f"{row['ORIGIN']}→{row['DEST']}"
    print(f"  {route:<14} {row['flights']:>6,}   {row['delay_rate']:>6.1%}   {row['avg_delay_min']:>6.1f}m"
          f"   {row['severe_delay_pct']:>6.1%}   {row['risk_score']:>5.3f} {row['risk_tier']}")

print(f"\n{'=' * 75}")
print(f"RISK TIER DISTRIBUTION")
print(f"{'=' * 75}")
tier_dist = route_stats.groupby('risk_tier').agg(
    routes=('risk_score', 'count'),
    avg_delay_rate=('delay_rate', 'mean'),
    total_flights=('flights', 'sum'),
)
for tier, row in tier_dist.iterrows():
    print(f"  {tier:<12} {row['routes']:>6} routes   {row['avg_delay_rate']:>6.1%} avg delay   {row['total_flights']:>10,} flights")

Routes with 50+ flights: 5,855

TOP 20 HIGHEST RISK ROUTES (min 50 flights)
  Route           Flights   Delay%  Avg Min  Severe%   Risk Tier      
  --------------------------------------------------------------------
  DFW→ALB           196    56.6%     61.8m    28.1%   0.605 High
  RIC→EWR           163    57.7%     61.4m    32.5%   0.601 High
  JAX→CVG            85    45.9%     68.9m    23.5%   0.594 Moderate
  DFW→SYR           196    57.1%     57.5m    26.0%   0.592 Moderate
  DCA→SAT           175    54.9%     51.8m    30.9%   0.589 Moderate
  PHL→BUF            82    48.8%     53.9m    26.8%   0.587 Moderate
  COS→SNA            55    43.6%     96.4m    23.6%   0.574 Moderate
  ORD→FAT           123    51.2%     45.1m    26.0%   0.548 Moderate
  RNO→JFK           122    36.1%     93.5m    23.0%   0.543 Moderate
  DCA→AUS           236    56.8%     42.1m    25.8%   0.532 Moderate
  DEN→ALB           203    53.2%     34.2m    20.7%   0.532 Moderate
  JAX→PIT            77    42.9

In [ ]:
route_stats.to_parquet('models/route_risk_scores.parquet')
print(f"✓ Saved: models/route_risk_scores.parquet ({len(route_stats):,} routes)")

print(f"\n{'=' * 60}")
print(f"ROUTE LOOKUP EXAMPLES:")
print(f"{'=' * 60}")

examples = [('ORD', 'LAX'), ('EWR', 'SFO'), ('ATL', 'JFK'), ('LIH', 'KOA')]
for orig, dest in examples:
    route = route_stats[(route_stats['ORIGIN'] == orig) & (route_stats['DEST'] == dest)]
    if len(route) > 0:
        r = route.iloc[0]
        print(f"\n  {orig}→{dest}: Risk {r['risk_score']:.3f} ({r['risk_tier']})")
        print(f"    Flights: {r['flights']:,} | Delay rate: {r['delay_rate']:.1%} | "
              f"Avg delay: {r['avg_delay_min']:.1f} min | Severe: {r['severe_delay_pct']:.1%}")
    else:
        print(f"\n  {orig}→{dest}: Route not found (< 50 flights)")

print(f"\n{'=' * 60}")
print(f"STEP 9 COMPLETE")
print(f"  5,855 routes scored")
print(f"  Risk components: delay rate, severity, cascade, weather, congestion")
print(f"  Ready for Streamlit route risk page")
print(f"{'=' * 60}")

✓ Saved: models/route_risk_scores.parquet (5,855 routes)

ROUTE LOOKUP EXAMPLES:

  ORD→LAX: Risk 0.262 (Low)
    Flights: 4,612 | Delay rate: 21.9% | Avg delay: 5.0 min | Severe: 7.8%

  EWR→SFO: Risk 0.282 (Low)
    Flights: 2,830 | Delay rate: 24.7% | Avg delay: 10.2 min | Severe: 11.7%

  ATL→JFK: Risk 0.334 (Low)
    Flights: 2,617 | Delay rate: 29.3% | Avg delay: 18.8 min | Severe: 12.6%

  LIH→KOA: Risk 0.273 (Low)
    Flights: 238 | Delay rate: 23.1% | Avg delay: 6.6 min | Severe: 2.5%

STEP 9 COMPLETE
  5,855 routes scored
  Risk components: delay rate, severity, cascade, weather, congestion
  Ready for Streamlit route risk page
